# 10 — IoU Threshold Sensitivity: τ=0.50 vs τ=0.25 (DIAGNOSTIC)

> **Justifies the per-city IoU threshold** the pipeline applies (τ=0.25 for
> SpaceNet cities, τ=0.50 for others). **Scope: SpaceNet cities** (raw building
> files available on Drive).

Compares city-level F1 for the SpaceNet cohort at:

| Threshold | Meaning |
|-----------|---------|
| **τ = 0.50** | PASCAL-VOC standard (strict) |
| **τ = 0.25** | SpaceNet benchmark standard (lenient) |

**Both thresholds are recomputed from raw geometries.** Since the pipeline now
applies a *per-city* threshold, the stored tile metrics for these SpaceNet
cities are already at τ=0.25 and cannot serve as a τ=0.50 baseline — so trusting
them would give Δ ≡ 0. Recomputing both from raw makes the comparison
self-consistent. All other parameters are held constant (buffer = `tau_buffer_m`,
min area, CRS).

**Efficiency:** matching is greedy 1-to-1 by descending IoU, so one match at
τ=0.25 yields both thresholds exactly (τ=0.50 = the τ=0.25 matches truncated to
IoU ≥ 0.50).

**Why this matters:** the matched-pair IoU distribution (final figure) shows the
fraction of *true* matches sitting in [0.25, 0.50) — accuracy that τ=0.50
discards, largely due to SN7 reference boundary uncertainty (Planet imagery).

**Outputs:**
- `outputs/sensitivity_studies/iou_threshold_sensitivity.csv` (long: city × dataset × threshold)
- `outputs/sensitivity_studies/iou_threshold_sensitivity_wide.csv`
- figures in `outputs/scratch/`

In [ ]:
!pip install -q geopandas shapely
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load τ=0.50 baseline from stored tile metrics ────────────
import sys, time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg.get('data_dir', 'data/01_raw')
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
SENS_DIR     = PROJECT_ROOT / 'outputs' / 'sensitivity_studies'
SENS_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

vec_pre        = cfg['vector']['preprocessing']
TAU_PIPELINE   = float(vec_pre.get('iou_threshold', vec_pre.get('tau_overlap', 0.5)))
TAU_BUFFER_M   = float(vec_pre.get('tau_buffer_m', 2.0))
MIN_AREA_M2    = float(vec_pre.get('min_area_m2', 20.0))
FIX_GEOMS      = bool(vec_pre.get('fix_invalid_geoms', True))
TAU_COMPARE    = 0.25   # SpaceNet benchmark standard

TILE_SENTINEL  = 'vector_metrics_tiles_all_datasets.parquet'

print(f'Pipeline IoU threshold : {TAU_PIPELINE}')
print(f'Comparison threshold   : {TAU_COMPARE}')
print(f'Buffer                 : {TAU_BUFFER_M} m  (kept identical for both runs)')

# ── SpaceNet7 flag ────────────────────────────────────────────────────────────
TRACKER_PATH = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
ref_source_map = {}
ref_file_map   = {}

if TRACKER_PATH.exists():
    tracker = pd.read_csv(TRACKER_PATH, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    id_col = 'dataset_folder_name'
    if 'reference_source' in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=[id_col])
            .set_index(id_col)['reference_source']
            .str.strip().str.lower().to_dict()
        )
    ref_col = next(
        (c for c in tracker.columns if 'reference' in c.lower() and 'file' in c.lower()), None
    )
    if ref_col:
        for _, row in tracker.dropna(subset=[id_col]).iterrows():
            city = str(row[id_col]).strip()
            raw  = str(row.get(ref_col, '') or '')
            parts = [p.strip() for p in raw.split('|') if p.strip()]
            if parts:
                ref_file_map.setdefault(city, []).extend(parts)
    sn7 = sum(1 for v in ref_source_map.values() if v == 'spacenet')
    print(f'Tracker loaded: {len(ref_source_map)} cities  ({sn7} SpaceNet7)')
else:
    print(f'[WARN] Tracker not found at {TRACKER_PATH}')

# ── Restrict to SpaceNet cities ───────────────────────────────────────────────
# τ=0.25 is the SpaceNet benchmark standard; the comparison is most meaningful
# for cities with SpaceNet7 reference data. Non-SpaceNet cities are also the
# ones most likely to be missing raw building files on Drive.
CITY_SUBSET = sorted(c for c, src in ref_source_map.items() if src == 'spacenet')
if CITY_SUBSET:
    print(f'\nRunning for {len(CITY_SUBSET)} SpaceNet cities: {CITY_SUBSET}')
else:
    print('\n[WARN] No SpaceNet cities found in tracker — falling back to all cities.')
    CITY_SUBSET = None

# ── Load τ=0.50 baseline from stored tile metrics ─────────────────────────────
baseline_rows = []
city_dirs_all = sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL))
for city_dir in city_dirs_all:
    city = city_dir.name
    if CITY_SUBSET and city not in CITY_SUBSET:
        continue
    try:
        tile_df = pd.read_parquet(city_dir / TILE_SENTINEL)
        for ds, g in tile_df.groupby('dataset'):
            tp = int(g['tp'].sum()); fp = int(g['fp'].sum()); fn = int(g['fn'].sum())
            p  = tp / (tp + fp) if (tp + fp) else 0.0
            r  = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2*p*r / (p+r) if (p+r) else 0.0
            baseline_rows.append({
                'city': city, 'dataset': ds,
                'is_spacenet7': True,
                f'f1_iou{int(TAU_PIPELINE*100):02d}': round(f1, 4),
            })
    except Exception as e:
        print(f'  [WARN] {city}: {e}')

df_baseline = pd.DataFrame(baseline_rows)
print(f'\nBaseline (τ={TAU_PIPELINE}): {len(df_baseline)} rows | {df_baseline["city"].nunique()} cities')
print('Cell 1 done.')

In [ ]:
# ── Cell 2 — Recompute F1 at BOTH τ=0.50 and τ=0.25 from raw building files ───
#
# WHY recompute both: the pipeline now applies a PER-CITY IoU threshold (τ=0.25
# for SpaceNet cities, τ=0.50 for others). So the stored tile metrics for the
# SpaceNet cohort studied here are ALREADY at τ=0.25 and cannot serve as a
# τ=0.50 baseline — trusting them makes Δ ≡ 0. We recompute BOTH thresholds
# from raw geometries, so the comparison is self-consistent regardless of what
# the pipeline stored.
#
# Efficiency: match_buildings_iou is greedy 1-to-1 by descending IoU, so a
# single match at the lower τ yields both thresholds exactly. Pairs with
# IoU ≥ τ_hi are matched identically at both; a matched pair in [τ_lo, τ_hi)
# has no ≥ τ_hi partner, so raising the threshold only drops it (ref→FN,
# cand→FP). One matching pass per tile; F1 at τ_hi derived by counting
# matched pairs with IoU ≥ τ_hi.

from src.metrics.vector.matching import match_buildings_iou
from src.utils.buildings import load_buildings
from src.utils.tiling import subset_by_tile

TAU_LENIENT = TAU_COMPARE   # 0.25 — SpaceNet benchmark standard
TAU_STRICT  = 0.50          # PASCAL-VOC standard

skipped     = []
recomp_rows = []
match_ious  = []   # every matched-pair IoU (≥ τ_lo); drives the distribution plot

cities_to_run = df_baseline['city'].unique().tolist()
print(f'Recomputing F1 at τ={TAU_LENIENT} and τ={TAU_STRICT} from raw for {len(cities_to_run)} cities...')
print(f'(buffer kept at {TAU_BUFFER_M} m; single matching pass per tile)\n')


def find_reference_files(city):
    if city in ref_file_map:
        paths = [DATA_DIR / city / 'vector' / f for f in ref_file_map[city]]
        return [p for p in paths if p.exists()]
    # fallback: any gpkg in vector/ that isn't a known candidate
    vec_dir = DATA_DIR / city / 'vector'
    if not vec_dir.exists():
        return []
    cand_names = {'overture', 'gba', 'globfp'}
    return [f for f in vec_dir.glob('*.gpkg')
            if not any(n in f.stem.lower() for n in cand_names)]


def find_candidate_files(city, ds_name):
    city_slug = city.lower()
    vec_dir   = DATA_DIR / city / 'vector'
    pattern   = f'{city_slug.replace("-", "_")}_{ds_name}*.parquet'
    return sorted(vec_dir.glob(pattern))


def city_f1_dual_tau(ref_all, cand_all, tiles, tau_lo, tau_hi, iou_sink=None):
    """Match ONCE at tau_lo; return (f1_lo, f1_hi) derived exactly at both.

    Relies on match_buildings_iou being greedy 1-to-1 by descending IoU (so the
    tau_hi matching is the tau_lo matching truncated to IoU >= tau_hi).
    """
    ref_idx  = ref_all.sindex
    cand_idx = cand_all.sindex
    tp_lo = fp_lo = fn_lo = 0
    tp_hi = fp_hi = fn_hi = 0
    for tile_row in tiles.itertuples():
        ref_tile  = subset_by_tile(ref_all,  ref_idx,  tile_row.geometry)
        cand_tile = subset_by_tile(cand_all, cand_idx, tile_row.geometry)
        if ref_tile.empty and cand_tile.empty:
            continue
        n_ref, n_cand = len(ref_tile), len(cand_tile)
        matches, _, _ = match_buildings_iou(
            ref_tile, cand_tile, tau_lo, tau_buffer_m=TAU_BUFFER_M
        )
        m_lo = len(matches)
        if m_lo:
            ious = matches['iou'].to_numpy()
            m_hi = int((ious >= tau_hi).sum())
            if iou_sink is not None:
                iou_sink.extend(ious.tolist())
        else:
            m_hi = 0
        tp_lo += m_lo; fp_lo += n_cand - m_lo; fn_lo += n_ref - m_lo
        tp_hi += m_hi; fp_hi += n_cand - m_hi; fn_hi += n_ref - m_hi

    def _f1(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        return round(2 * p * r / (p + r) if (p + r) else 0.0, 4)

    return _f1(tp_lo, fp_lo, fn_lo), _f1(tp_hi, fp_hi, fn_hi)


for city in cities_to_run:
    t0 = time.time()
    ref_paths = find_reference_files(city)
    if not ref_paths:
        skipped.append((city, 'reference files not found'))
        continue

    city_slug  = city.lower()
    tiles_path = DATA_DIR / city / 'tiles' / f'{city_slug}_tiles.gpkg'
    if not tiles_path.exists():
        skipped.append((city, 'tiles GPKG not found'))
        continue

    try:
        tiles = gpd.read_file(tiles_path)
        crs   = tiles.crs.to_string()
    except Exception as e:
        skipped.append((city, f'tiles load error: {e}'))
        continue

    try:
        ref_parts = [load_buildings(p, crs_work=crs, min_area_m2=MIN_AREA_M2,
                                    fix_invalid_geoms=FIX_GEOMS) for p in ref_paths]
        ref_all = (gpd.GeoDataFrame(pd.concat(ref_parts, ignore_index=True),
                                    crs=ref_parts[0].crs)
                   if len(ref_parts) > 1 else ref_parts[0])
    except Exception as e:
        skipped.append((city, f'reference load error: {e}'))
        continue

    ds_names = df_baseline[df_baseline['city'] == city]['dataset'].unique()
    is_sn7   = ref_source_map.get(city) == 'spacenet'
    any_ran  = False
    for ds_name in ds_names:
        cand_files = find_candidate_files(city, ds_name)
        if not cand_files:
            continue
        try:
            cand_all = load_buildings(cand_files[0], crs_work=crs,
                                      min_area_m2=MIN_AREA_M2, fix_invalid_geoms=FIX_GEOMS)
        except Exception as e:
            print(f'  [WARN] {city}/{ds_name}: {e}')
            continue

        f1_lenient, f1_strict = city_f1_dual_tau(
            ref_all, cand_all, tiles, TAU_LENIENT, TAU_STRICT, iou_sink=match_ious
        )
        recomp_rows.append({
            'city': city, 'dataset': ds_name,
            'f1_iou25': f1_lenient, 'f1_iou50': f1_strict,
            'is_spacenet7': is_sn7,
        })
        any_ran = True

    elapsed = time.time() - t0
    print(f'  {city:<40} {"ok" if any_ran else "no candidates"}  ({elapsed:.0f}s)')

df_recomp = pd.DataFrame(recomp_rows)

if skipped:
    print(f'\nSkipped {len(skipped)} cities (raw data not on Drive):')
    for city, reason in skipped[:5]:
        print(f'  {city}: {reason}')
    if len(skipped) > 5:
        print(f'  ... and {len(skipped)-5} more')

print(f'\nRecomputed both τ from raw: {len(df_recomp)} rows | {len(match_ious):,} matched pairs (IoU ≥ {TAU_LENIENT})')

In [ ]:
# ── Cell 3 — Comparison table + plots ────────────────────────────────────────

# Both thresholds are recomputed from raw geometries in Cell 2 (df_recomp);
# the stored baseline is no longer used for the delta.
df = df_recomp.copy()
df['delta_025_vs_50'] = (df['f1_iou25'] - df['f1_iou50']).round(4)

GROUP_PALETTE = {'SpaceNet7': '#0072B2', 'Non-SpaceNet': '#E69F00'}
df['group'] = df['is_spacenet7'].map({True: 'SpaceNet7', False: 'Non-SpaceNet'})

# ── Summary table ─────────────────────────────────────────────────────────────
print('=== F1 at τ=0.50 (strict) vs τ=0.25 (lenient), both recomputed from raw ===')
df_city = (
    df.groupby(['city', 'is_spacenet7', 'group'])[
        ['f1_iou50', 'f1_iou25', 'delta_025_vs_50']
    ].mean().reset_index().round(4)
)
print(f'Cities with both values: {df_city["f1_iou25"].notna().sum()} / {len(df_city)}\n')
for grp_label, grp in df_city.groupby('group'):
    grp_valid = grp.dropna(subset=['f1_iou25'])
    print(f'  {grp_label} ({len(grp_valid)} cities):')
    for col in ['f1_iou50', 'f1_iou25', 'delta_025_vs_50']:
        if grp_valid[col].notna().any():
            print(f'    {col:<22}  mean={grp_valid[col].mean():.4f}  '
                  f'median={grp_valid[col].median():.4f}  std={grp_valid[col].std():.4f}')
    print()

# ── Plots ─────────────────────────────────────────────────────────────────────
df_plot = df_city.dropna(subset=['f1_iou25'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: F1 at τ=0.50 vs τ=0.25 side by side
df_melt = df_plot.melt(
    id_vars=['city', 'group'],
    value_vars=['f1_iou50', 'f1_iou25'],
    var_name='threshold', value_name='f1'
)
df_melt['threshold_label'] = df_melt['threshold'].map(
    {'f1_iou50': 'τ = 0.50\n(strict)', 'f1_iou25': 'τ = 0.25\n(lenient)'})

sns.boxplot(data=df_melt, x='threshold_label', y='f1', hue='group',
            palette=GROUP_PALETTE, ax=axes[0], linewidth=1.2)
axes[0].set_title('F1 at τ=0.50 (strict) vs τ=0.25 (lenient)', fontweight='bold')
axes[0].set_xlabel('IoU threshold')
axes[0].set_ylabel('City-level F1')
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Group', fontsize=8)

# Panel 2: Δ F1 (τ=0.25 minus τ=0.50) — how much the more lenient threshold gains
sns.boxplot(data=df_plot, x='group', y='delta_025_vs_50',
            palette=GROUP_PALETTE, ax=axes[1], linewidth=1.2)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Δ F1: τ=0.25 minus τ=0.50\n(positive = lenient threshold gains F1)',
                  fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('ΔF1 (τ0.25 − τ0.50)')

# Panel 3: Scatter τ=0.50 vs τ=0.25 — one dot per city × dataset
DATASET_MARKERS = {'overture': 'o', 'gba': 's', 'globfp': '^'}
df_scatter = df.dropna(subset=['f1_iou25'])
for ds, ds_grp in df_scatter.groupby('dataset'):
    for grp_label, g in ds_grp.groupby('group'):
        axes[2].scatter(
            g['f1_iou50'], g['f1_iou25'],
            color=GROUP_PALETTE[grp_label],
            marker=DATASET_MARKERS.get(ds, 'o'),
            alpha=0.4, s=14,
            label=f'{ds}' if grp_label == 'SpaceNet7' else '_nolegend_'
        )
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='y = x (no change)')
axes[2].set_xlabel('F1 at τ = 0.50')
axes[2].set_ylabel('F1 at τ = 0.25')
axes[2].set_title('Scatter per city × dataset\n(above y=x → τ=0.25 gives higher F1)',
                  fontweight='bold')
axes[2].legend(fontsize=8, title='Dataset')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)

fig.suptitle(
    f'IoU threshold sensitivity — τ=0.50 vs τ=0.25, both recomputed from raw  '
    f'(buffer = {TAU_BUFFER_M} m, held constant)',
    fontsize=13, fontweight='bold', y=1.01
)
fig.tight_layout()
out_fig = SCRATCH_DIR / 'iou_threshold_sensitivity_boxplot.png'
fig.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out_fig}')

# ── Evidence: distribution of matched-pair IoU ────────────────────────────────
# This is what justifies τ=0.25 for SN7 reference: the fraction of TRUE matches
# whose IoU falls in [0.25, 0.50) is exactly the F1 that τ=0.50 discards.
import numpy as np
_ious = np.asarray(match_ious, dtype=float)
if _ious.size:
    frac_lenient_band = float(((_ious >= 0.25) & (_ious < 0.50)).mean())
    fig2, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(_ious, bins=np.arange(0.25, 1.001, 0.025), color='#0072B2', alpha=0.8)
    ax.axvline(0.50, color='#D55E00', linestyle='--', linewidth=1.5, label='τ = 0.50')
    ax.axvline(0.25, color='grey',    linestyle=':',  linewidth=1.2, label='τ = 0.25')
    ax.axvspan(0.25, 0.50, color='#D55E00', alpha=0.08)
    ax.set_title(
        f'Matched-pair IoU distribution (τ=0.25 matches, {len(_ious):,} pairs)\n'
        f'{frac_lenient_band:.1%} of true matches fall in [0.25, 0.50) — dropped by τ=0.50',
        fontweight='bold')
    ax.set_xlabel('IoU of matched reference–candidate pair')
    ax.set_ylabel('Matched pairs')
    ax.legend()
    sns.despine(ax=ax)
    fig2.tight_layout()
    out_fig2 = SCRATCH_DIR / 'iou_matched_pair_distribution.png'
    fig2.savefig(out_fig2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved → {out_fig2}')
    print(f'{frac_lenient_band:.1%} of matched pairs have 0.25 ≤ IoU < 0.50')
else:
    print('[skip] no matched pairs collected — cannot plot IoU distribution')

In [ ]:
# ── Cell 4 — Save CSV ─────────────────────────────────────────────────────────
# 07_visualizations (S1/S2) expects LONG format: one row per city x dataset x
# threshold, with iou_threshold + f1 columns. Write that as the canonical CSV and
# keep the wide table alongside for reading by eye.
_long = df.melt(
    id_vars=['city', 'dataset', 'is_spacenet7'],
    value_vars=['f1_iou50', 'f1_iou25'],
    var_name='_thr', value_name='f1',
).dropna(subset=['f1'])
_long['iou_threshold'] = _long['_thr'].map({'f1_iou50': 0.50, 'f1_iou25': 0.25})
_long = _long[['city', 'dataset', 'iou_threshold', 'f1', 'is_spacenet7']]
out_path = SENS_DIR / 'iou_threshold_sensitivity.csv'
_long.to_csv(out_path, index=False)
out_wide = SENS_DIR / 'iou_threshold_sensitivity_wide.csv'
df.to_csv(out_wide, index=False)

print(f'Saved → {out_path}')
print(f'  {len(_long):,} rows (long)  |  columns: {list(_long.columns)}')
print(f'  wide table -> {out_wide}')
print(f'  Cities with τ=0.25 data : {df["f1_iou25"].notna().sum() // df["dataset"].nunique()}')
print(f'  Cities skipped (no data): {len(skipped)}')
print()
print('=== Global means by dataset ===')
display(
    df.dropna(subset=['f1_iou25'])
    .groupby('dataset')[['f1_iou50', 'f1_iou25', 'delta_025_vs_50']]
    .mean().round(4)
)